# Corpus Statistics Analysis

This notebook extracts and analyzes statistics from the annotated quotation corpus, including overlap with the NeWMe (Negotiation of Meaning) corpus.
Provide the path to your copy of newme in `path_to_newme_json`.

We look into:

- How many instances were discarded?
- Distribution of high-level and fine-grained quotation types: SQ, NSQ, and both
- Frequency of "Unsure" and "Ambiguous" annotations
- Types and frequency of double annotations (label combinations)
- Label analysis for WMN instances and their corresponding span types
- Relationship between types of quotation marks (single vs. double quotes) and quotation functions
- Relationship between n-gram length (1, 2, 3) and quotation types



In [1]:
# Core libraries
import json
import re
from collections import Counter
from copy import deepcopy

# Data analysis
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from utils import get_all_labels, get_labels_one_level, get_conv_id, HIGH_LEVEL, FINE_LEVEL

path_to_newme_json = "SPECIFY_YOUR_PATH_HERE"  # Path to the NewMe JSON file containing annotation data


## Data Loading

In [2]:
# Load the main corpus of expert annotations
full_corpus = json.load(open("annotated_corpus.json", "r"))
print(f"Loaded {len(full_corpus)} annotated instances from corpus")

Loaded 2522 annotated instances from corpus


## Data Preprocessing

#### Removing Discarded Instances

In [3]:
# Filter out instances marked as "Discard"
corpus = [item for item in full_corpus if "Discard" not in get_all_labels(item)]

discarded_count = len(full_corpus) - len(corpus)

print("=" * 70)
print(f"📊 Corpus size after removing discarded instances: {len(corpus)}")
print(f"❌ Discarded instances: {discarded_count} ({discarded_count/len(full_corpus)*100:.1f}%)")
print("=" * 70)

📊 Corpus size after removing discarded instances: 2500
❌ Discarded instances: 22 (0.9%)


#### Enrich with information about quote symbols and ngrams 

In [4]:
quotes_all_info = pd.read_csv("obtaining_data/quotations_winningargs_info_public.tsv", sep="\t")
print(quotes_all_info.columns)

Index(['quoted_passage', 'ngram', 'utt_id', 'conv_id', 'author', 'start',
       'end', 'conv_in_newme', 'sign', 'item_order',
       'used_previously_in_subthread', 'id', 'kept_for_annotation'],
      dtype='object')


In [5]:
for item in corpus:
    other_item = quotes_all_info[quotes_all_info['item_order'] == item['data']['item_order']]
    assert len(other_item) == 1
    item['data']['sign'] = other_item['sign'].iloc[0]#.value
    item['data']['ngram'] = other_item['ngram'].iloc[0]#.value
    item['data']['start'] = other_item['start'].iloc[0]#.value
    item['data']['end'] = other_item['end'].iloc[0]#.value
    

---

## High-Level Label Distribution

How many instances are labeled as SQ (Standard Quotation), NSQ (Non-Standard Quotation), or both?

In [6]:
high_level_labels = [get_labels_one_level(item, HIGH_LEVEL) for item in corpus]

c = Counter(high_level_labels)

print("\n" + "=" * 70)
print("HIGH-LEVEL LABEL DISTRIBUTION (including Unsure/Ambiguous)")
print("=" * 70)
for k in sorted(c.keys()):
    pct = c[k]/len(corpus)*100
    print(f"{str(k):50s} {c[k]:4d} ({pct:5.1f}%)")


HIGH-LEVEL LABEL DISTRIBUTION (including Unsure/Ambiguous)
('Non-scare quotes',)                               899 ( 36.0%)
('Non-scare quotes', 'Unsure')                       17 (  0.7%)
('Scare quotes',)                                  1214 ( 48.6%)
('Scare quotes', 'Non-scare quotes')                264 ( 10.6%)
('Scare quotes', 'Non-scare quotes', 'Ambiguous')    67 (  2.7%)
('Scare quotes', 'Non-scare quotes', 'Ambiguous', 'Unsure')   20 (  0.8%)
('Scare quotes', 'Non-scare quotes', 'Unsure')       11 (  0.4%)
('Scare quotes', 'Unsure')                            8 (  0.3%)


### Uncertain and Ambiguous Annotations at High Level

In [7]:
total_unsures = sum([c[lbs] for lbs in c if "Unsure" in lbs])

print(f"\n⚠️  Instances with 'Unsure' at high level: {total_unsures} ({np.round(total_unsures/len(corpus)*100, 1)}%)")


total_ambiguous = sum([c[lbs] for lbs in c if "Ambiguous" in lbs])

print(f"\n⚠️  Instances with 'Ambiguous' at high level: {total_ambiguous} ({np.round(total_ambiguous/len(corpus)*100, 1)}%)")


⚠️  Instances with 'Unsure' at high level: 56 (2.2%)

⚠️  Instances with 'Ambiguous' at high level: 87 (3.5%)


### High-Level Distribution ignoring Unsure/Ambiguous

In [8]:
high_level_labels_ignore_unsures = [
    tuple([lb for lb in labels if lb not in ["Unsure", "Ambiguous"]]) 
    for labels in high_level_labels
]

c = Counter(high_level_labels_ignore_unsures)

print("\n" + "=" * 70)
print("HIGH-LEVEL DISTRIBUTION (ignoring Unsure/Ambiguous)")
print("=" * 70)

for k in sorted(c.keys()):
    pct = c[k]/len(corpus)*100
    print(f"{str(k):50s} {c[k]:4d} ({pct:5.1f}%)")


HIGH-LEVEL DISTRIBUTION (ignoring Unsure/Ambiguous)
('Non-scare quotes',)                               916 ( 36.6%)
('Scare quotes',)                                  1222 ( 48.9%)
('Scare quotes', 'Non-scare quotes')                362 ( 14.5%)


---

### "Unsure" annotations at any level

How many instances have "Unsure" labels at either high or fine level?

In [9]:
involves_unsure = 0
confusing = []

for item in corpus:
    hlls = get_labels_one_level(item, HIGH_LEVEL)
    flls = get_labels_one_level(item, FINE_LEVEL)
    if "Unsure" in hlls or "Unsure" in flls:
        involves_unsure += 1
        if "Unsure" in flls:            
            confusing.append(flls)

pct_unsure = np.round(involves_unsure/len(corpus)*100, 1)
print("\n" + "=" * 70)
print(f"⚠️  Instances with 'Unsure' at ANY level: {involves_unsure} / {len(corpus)} ({pct_unsure}%)")
print("=" * 70)

counter = Counter(confusing)
print(f"\n📊 Fine-level label combinations involving 'Unsure' ({len(counter)} unique combinations):")
for combi, count in counter.most_common(10):
    print(f"  {str(combi):60s} {count:3d}")


⚠️  Instances with 'Unsure' at ANY level: 84 / 2500 (3.4%)

📊 Fine-level label combinations involving 'Unsure' (22 unique combinations):
  ('NSQ: Meta-linguistic comment', 'Unsure')                     5
  ('NSQ: Emphasis', 'Unsure')                                    4
  ('SQ: Usage', 'NSQ: Designation', 'Unsure')                    3
  ('NSQ: Designation', 'Unsure')                                 3
  ('SQ: Usage', 'NSQ: Emphasis', 'Unsure')                       2
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Meta-linguistic comment', 'Unsure')   2
  ('SQ: Usage', 'SQ: Echo Flag', 'Unsure')                       2
  ('SQ: Usage', 'Unsure')                                        2
  ('NSQ: Direct quoting', 'Unsure')                              2
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Designation', 'Unsure')   2


---

## Fine-grained Label Distribution

Detailed analysis of quotation subtypes and their combinations (including all flags)

In [10]:
fine_level_labels = [get_labels_one_level(item, FINE_LEVEL) for item in corpus]

c = Counter(fine_level_labels)

print("\n" + "=" * 70)
print("MOST COMMON FINE-LEVEL LABELS AND COMBINATIONS (All labels included)")
print("=" * 70)
print(f"Total unique label combinations: {len(c)}\n")

# Show top 20 most common combinations
for k, count in c.most_common(20):
    pct = count/len(corpus)*100
    print(f"{str(k):60s} {count:4d} ({pct:5.1f}%)")


MOST COMMON FINE-LEVEL LABELS AND COMBINATIONS (All labels included)
Total unique label combinations: 65

('SQ: Usage',)                                                756 ( 30.2%)
('NSQ: Meta-linguistic comment',)                             443 ( 17.7%)
('SQ: Usage', 'SQ: Echo Flag')                                366 ( 14.6%)
('NSQ: Designation',)                                         271 ( 10.8%)
('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Meta-linguistic comment')  172 (  6.9%)
('NSQ: Direct quoting',)                                      109 (  4.4%)
('SQ: Usage', 'NSQ: Designation')                              62 (  2.5%)
('SQ: Usage', 'NSQ: Meta-linguistic comment')                  52 (  2.1%)
('SQ: Mixed quotation with attitude',)                         43 (  1.7%)
('SQ: Word',)                                                  29 (  1.2%)
('NSQ: Emphasis',)                                             27 (  1.1%)
('NSQ: Mixed quoting without attitude',)                       22 

### Fine-Level Labels (Ignoring Unsure, Echo, and Ambiguous)

Combinations without uncertain, ambiguous or echo annotations:

In [11]:
fine_level_labels_ignore_unsures = [tuple(sorted([lb for lb in labels if lb not in ["Unsure","SQ: Echo Flag"] and "Ambiguous" not in lb])) for labels in fine_level_labels]

c = Counter(fine_level_labels_ignore_unsures)
c_sorted = dict(sorted(c.items(), key=lambda item: item[1], reverse=True))

num_single = len([k for k in c if len(k) == 1])
num_combos = len([k for k in c if len(k) > 1])

print("\n" + "=" * 70)
print("FINE-LEVEL LABEL COMBINATIONS (ignoring Unsure/FLAG/Ambiguous)")
print("=" * 70)
print(f"\n📊 Summary:")
print(f"  Total unique labelings: {len(c):3d}")
print(f"  Single labels:          {num_single:3d}")
print(f"  Multiple label combos:  {num_combos:3d}")

print("\n" + "-" * 70)
print("Most frequent combinations:")
for i, (k, count) in enumerate(c_sorted.items(), 1):
    pct = count/len(corpus)*100
    marker = "📌" if len(k) > 1 else "  "
    if i <= 30:  # Show top 30
        print(f"{marker} {str(k):58s} {count:4d} ({pct:5.2f}%)")
print("=" * 70)


FINE-LEVEL LABEL COMBINATIONS (ignoring Unsure/FLAG/Ambiguous)

📊 Summary:
  Total unique labelings:  28
  Single labels:            8
  Multiple label combos:   20

----------------------------------------------------------------------
Most frequent combinations:
   ('SQ: Usage',)                                             1127 (45.08%)
   ('NSQ: Meta-linguistic comment',)                           448 (17.92%)
   ('NSQ: Designation',)                                       274 (10.96%)
📌 ('NSQ: Meta-linguistic comment', 'SQ: Usage')               227 ( 9.08%)
   ('NSQ: Direct quoting',)                                    111 ( 4.44%)
📌 ('NSQ: Designation', 'SQ: Usage')                            80 ( 3.20%)
   ('SQ: Mixed quotation with attitude',)                       63 ( 2.52%)
   ('SQ: Word',)                                                32 ( 1.28%)
   ('NSQ: Emphasis',)                                           31 ( 1.24%)
   ('NSQ: Mixed quoting without attitude',)         

### Individual Label Frequencies (Pure vs. Combined)

Breakdown of how often each label appears alone vs. in combination with others (ignoring Echo):

(Table 3 of the paper)

In [12]:
fine_level_labels_ignore_unsures = [tuple(sorted([lb for lb in labels if lb not in ["Unsure"] and "Ambiguous" not in lb and "FLAG" not in lb])) for labels in fine_level_labels]

c = Counter(fine_level_labels_ignore_unsures)

total_diff_individual_labels = sorted(set([lb for lbcombi in c.keys() for lb in lbcombi]))

print("\n" + "=" * 70)
print("INDIVIDUAL LABEL FREQUENCIES")
print("=" * 70)
print(f"{'Label':<50} {'Pure':>6} {'Combined':>9} {'Total':>7} {'Pct':>7}")
print("-" * 70)

for lb in total_diff_individual_labels:
    pure = c.get((lb,), 0)
    combined = sum([c[k] for k in c.keys() if lb in k and k != (lb,)])
    total_lb = pure + combined
    pct = (total_lb/len(corpus)*100)
    print(f"{lb:<50} {pure:6d} {combined:9d} {total_lb:7d} {pct:6.1f}%")

print("=" * 70)


INDIVIDUAL LABEL FREQUENCIES
Label                                                Pure  Combined   Total     Pct
----------------------------------------------------------------------
NSQ: Designation                                      274       113     387   15.5%
NSQ: Direct quoting                                   111         6     117    4.7%
NSQ: Emphasis                                          31        27      58    2.3%
NSQ: Meta-linguistic comment                          448       269     717   28.7%
NSQ: Mixed quoting without attitude                    22        20      42    1.7%
SQ: Echo Flag                                           0       599     599   24.0%
SQ: Mixed quotation with attitude                      44        36      80    3.2%
SQ: Usage                                             759       705    1464   58.6%
SQ: Word                                               30        10      40    1.6%


#### Uncertain annotations and the fine-grained level

In [13]:
# Count instances with Unsure, Ambiguous, and FLAG labels at fine level
flag_label = "SQ: Echo Flag"
c = Counter(fine_level_labels)

sum_unsures = sum([c[k] for k in c if "Unsure" in k])
total_ambiguous = 0
for k in fine_level_labels:   
    ambi_there = [lb for lb in k if "Ambiguous" in lb]
    if ambi_there:
        total_ambiguous += c[k]
sum_flags = sum([c[k] for k in c if flag_label in k])

print("\n" + "=" * 70)
print("UNCERTAIN / SPECIAL ANNOTATIONS AT FINE LEVEL")
print("=" * 70)
print(f"Instances with 'Unsure':    {sum_unsures:4d} ({sum_unsures/len(corpus)*100:5.1f}%)")
print(f"Instances with 'Ambiguous': {total_ambiguous:4d} ({total_ambiguous/len(corpus)*100:5.1f}%)")
print(f"Instances with FLAG:        {sum_flags:4d} ({sum_flags/len(corpus)*100:5.1f}%)")
print("=" * 70)


UNCERTAIN / SPECIAL ANNOTATIONS AT FINE LEVEL
Instances with 'Unsure':      39 (  1.6%)
Instances with 'Ambiguous':    6 (  0.2%)
Instances with FLAG:         599 ( 24.0%)


In [14]:
# Multiple label combinations (excluding Unsure, Ambiguous, and FLAG)
fine_level_labels_ignore_unsures_and_flag = [tuple(sorted([lb for lb in labels if lb not in ["Unsure"] and "Ambiguous" not in lb and "SQ: Echo Flag" not in lb])) for labels in fine_level_labels]

c = Counter(fine_level_labels_ignore_unsures_and_flag)
c_sorted = dict(sorted(c.items(), key=lambda item: item[1], reverse=True))

combos = [(k, count) for k, count in c_sorted.items() if len(k) > 1]
instances_with_combis = sum([count for k, count in combos])

print("\n" + "=" * 70)
print("MULTIPLE LABEL COMBINATIONS (Core labels only, no Echo/Unsure/Ambiguous)")
print("=" * 70)
print(f"\n📊 Summary:")
print(f"  Total unique combination patterns: {len(combos):3d}")
print(f"  Instances with multiple labels:    {instances_with_combis:4d} ({instances_with_combis/len(corpus)*100:5.1f}%)")

# Distribution by combination length
combis_lens = [len(k) for k, _ in combos]
len_dist = Counter(combis_lens)
print(f"\n  Combination lengths:")
for length in sorted(len_dist.keys()):
    print(f"    {length} labels: {len_dist[length]:3d} patterns")

print("\n" + "-" * 70)
print("Most frequent combinations:")
for i, (k, count) in enumerate(combos[:20], 1):  # Top 20
    pct = count/len(corpus)*100
    print(f"  {i:2d}. {str(k):55s} {count:4d} ({pct:5.2f}%)")

print("=" * 70)


MULTIPLE LABEL COMBINATIONS (Core labels only, no Echo/Unsure/Ambiguous)

📊 Summary:
  Total unique combination patterns:  20
  Instances with multiple labels:     392 ( 15.7%)

  Combination lengths:
    2 labels:  15 patterns
    3 labels:   5 patterns

----------------------------------------------------------------------
Most frequent combinations:
   1. ('NSQ: Meta-linguistic comment', 'SQ: Usage')            227 ( 9.08%)
   2. ('NSQ: Designation', 'SQ: Usage')                         80 ( 3.20%)
   3. ('NSQ: Emphasis', 'SQ: Usage')                            17 ( 0.68%)
   4. ('NSQ: Designation', 'NSQ: Meta-linguistic comment')      13 ( 0.52%)
   5. ('NSQ: Mixed quoting without attitude', 'SQ: Mixed quotation with attitude')   12 ( 0.48%)
   6. ('NSQ: Designation', 'NSQ: Meta-linguistic comment', 'SQ: Usage')    8 ( 0.32%)
   7. ('NSQ: Designation', 'SQ: Word')                           5 ( 0.20%)
   8. ('NSQ: Meta-linguistic comment', 'SQ: Mixed quotation with attitude')    5 

---

## Label Distribution by type of quotation marks

Single quotes (') vs. double quotes (")

### High-Level Labels by Quote Type

In [15]:
high_level_labels_by_quotes = {'"':[],"'":[]}
high_level_labels_by_quotes_wunsure = {'"':[],"'":[]}

observations = []

# Without Unsure/Ambiguous
for item in corpus:
    lbs = get_labels_one_level(item, HIGH_LEVEL)
    lbs = tuple([lb for lb in lbs if lb not in ["Unsure"] and "Ambiguous" not in lb])    
    high_level_labels_by_quotes[item['data']['sign']].append(lbs)

print("\n" + "=" * 70)
print("HIGH-LEVEL LABELS BY QUOTE TYPE (ignoring Unsure/Ambiguous)")
print("=" * 70)
    
for sign in ['"', "'"]:
    quote_name = "Double quotes (\")" if sign == '"' else "Single quotes (')"
    print(f"\n{quote_name}: {len(high_level_labels_by_quotes[sign])} instances")
    print("-" * 50)
    
    c = Counter(high_level_labels_by_quotes[sign])
    
    o = []
    for k in sorted(c.keys()):
        pct = c[k]/sum(c.values())*100
        print(f"  {str(k):35s} {c[k]:4d} ({pct:5.1f}%)")
        o.append(c[k])
    observations.append(o)



HIGH-LEVEL LABELS BY QUOTE TYPE (ignoring Unsure/Ambiguous)

Double quotes ("): 2190 instances
--------------------------------------------------
  ('Non-scare quotes',)                814 ( 37.2%)
  ('Scare quotes',)                   1063 ( 48.5%)
  ('Scare quotes', 'Non-scare quotes')  313 ( 14.3%)

Single quotes ('): 310 instances
--------------------------------------------------
  ('Non-scare quotes',)                102 ( 32.9%)
  ('Scare quotes',)                    159 ( 51.3%)
  ('Scare quotes', 'Non-scare quotes')   49 ( 15.8%)


#### Statistical Significance Testing: High-Level Labels by Quote Type

In [16]:
stat, pval, dfr, exp = chi2_contingency(observations, correction=True)

print("\n" + "=" * 70)
print("CHI-SQUARE TEST: High-level labels by quote type")
print("=" * 70)
print(f"Test statistic: {stat:.4f}")
print(f"P-value: {pval:.6f}")
print(f"Degrees of freedom: {dfr}")

if pval < 0.05:
    print("\n✓ STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("  → Quote type (single vs double quotes) significantly affects high-level labels")
else:
    print("\n✗ NOT SIGNIFICANT (p ≥ 0.05)")
print("=" * 70)


CHI-SQUARE TEST: High-level labels by quote type
Test statistic: 2.1992
P-value: 0.332996
Degrees of freedom: 2

✗ NOT SIGNIFICANT (p ≥ 0.05)


In [17]:
fine_level_labels_by_quotes = {'"':[],"'":[]}

# Without Unsure/Ambiguous/FLAG
for item in corpus:
    lbs = get_labels_one_level(item, FINE_LEVEL)
    lbs = tuple([lb for lb in lbs if lb not in ["Unsure", flag_label] and "Ambiguous" not in lb])
    fine_level_labels_by_quotes[item['data']['sign']].append(lbs)

observations_by_label = dict()

print("\n" + "=" * 70)
print("FINE-LEVEL LABELS BY QUOTE TYPE (ignoring Unsure/Ambiguous/FLAG)")
print("=" * 70)

for sign in ['"', "'"]:
    quote_name = "Double quotes (\")" if sign == '"' else "Single quotes (')"
    print(f"\n{quote_name}:")
    print("-" * 70)

    c = Counter(fine_level_labels_by_quotes[sign])
    total_diff_individual_labels = sorted(set([lb for lbcombi in c.keys() for lb in lbcombi]))

    print(f"{'Label':<50} {'Pure':>6} {'Comb':>6} {'Total':>7} {'Pct':>7}")
    print("-" * 70)

    total = len([x for x in corpus if x['data']['sign'] == sign])
    for lb in total_diff_individual_labels:
        if lb not in observations_by_label:
            observations_by_label[lb] = []

        pure = c.get((lb,), 0)
        combined = sum([c[k] for k in c.keys() if lb in k and k != (lb,)])
        total_lb = pure + combined
        pct = (total_lb/total*100) if total > 0 else 0

        print(f"{lb:<50} {pure:6d} {combined:6d} {total_lb:7d} {pct:6.1f}%")

        observations_by_label[lb].append([pure+combined, total-(pure+combined)])
    print()

# FLAG shown separately
print("-" * 70)
print("SQ FLAG (shown separately):")
print("-" * 70)
if flag_label not in observations_by_label:
    observations_by_label[flag_label] = []
for sign in ['"', "'"]:
    quote_name = "Double quotes (\")" if sign == '"' else "Single quotes (')"
    total = len([x for x in corpus if x['data']['sign'] == sign])
    flag_count = sum(1 for item in corpus
                     if item['data']['sign'] == sign
                     and flag_label in get_labels_one_level(item, FINE_LEVEL))
    pct = (flag_count / total * 100) if total > 0 else 0
    print(f"  {quote_name:<22} {flag_count:4d} / {total:4d} ({pct:5.1f}%)")
    observations_by_label[flag_label].append([flag_count, total - flag_count])
print("=" * 70)


FINE-LEVEL LABELS BY QUOTE TYPE (ignoring Unsure/Ambiguous/FLAG)

Double quotes ("):
----------------------------------------------------------------------
Label                                                Pure   Comb   Total     Pct
----------------------------------------------------------------------
NSQ: Designation                                      245     91     336   15.3%
NSQ: Direct quoting                                   106      6     112    5.1%
NSQ: Emphasis                                          24     21      45    2.1%
NSQ: Meta-linguistic comment                          393    238     631   28.8%
NSQ: Mixed quoting without attitude                    20     20      40    1.8%
SQ: Mixed quotation with attitude                      59     16      75    3.4%
SQ: Usage                                             979    291    1270   58.0%
SQ: Word                                               25      6      31    1.4%


Single quotes ('):
----------------------

### Statistical Significance Testing: Fine-Level Labels by Quote Type

In [18]:
print("\n" + "=" * 70)
print("CHI-SQUARE TESTS: Fine-level labels by quote type")
print("=" * 70)

significant_results = []

for lb in sorted(observations_by_label.keys()):
    stat, pval, dfr, exp = chi2_contingency(observations_by_label[lb], correction=True)
    
    status = "✓ SIGNIFICANT" if pval < 0.05 else "  -"
    print(f"{status} | {lb:45s} | p = {pval:.6f}")
    
    if pval < 0.05:
        significant_results.append(lb)

print("=" * 70)
if significant_results:
    print(f"\n✓ {len(significant_results)} categories show significant differences by quote type:")
    for lb in significant_results:
        print(f"  • {lb}")
else:
    print("\n✗ No significant differences found")


CHI-SQUARE TESTS: Fine-level labels by quote type
  - | NSQ: Designation                              | p = 0.673444
✓ SIGNIFICANT | NSQ: Direct quoting                           | p = 0.009651
✓ SIGNIFICANT | NSQ: Emphasis                                 | p = 0.032380
  - | NSQ: Meta-linguistic comment                  | p = 0.746624
  - | NSQ: Mixed quoting without attitude           | p = 0.201032
  - | SQ: Echo Flag                                 | p = 0.050170
  - | SQ: Mixed quotation with attitude             | p = 0.127516
  - | SQ: Usage                                     | p = 0.140540
  - | SQ: Word                                      | p = 0.086890

✓ 2 categories show significant differences by quote type:
  • NSQ: Direct quoting
  • NSQ: Emphasis


---

## Label distribution by n-gram length



In [19]:
high_level_labels_by_ngram = {'1':[],"+1":[]}
high_level_labels_by_ngram_wunsure = {'1':[],"+1":[]}

observations = []

# Without Unsure/Ambiguous
for item in corpus:
    lbs = get_labels_one_level(item, HIGH_LEVEL)
    lbs = tuple([lb for lb in lbs if lb not in ["Unsure"] and "Ambiguous" not in lb])    
    ngram = '1' if item['data']['ngram'] == '1' else '+1'
    high_level_labels_by_ngram[ngram].append(lbs)

print("\n" + "=" * 70)
print("HIGH-LEVEL LABELS BY N-GRAM LENGTH (ignoring Unsure/Ambiguous)")
print("=" * 70)
    
for ngram in ['1', '+1']:
    ngram_name = "1-grams (single words)" if ngram == '1' else "+1-grams (multi-word phrases)"
    print(f"\n{ngram_name}: {len(high_level_labels_by_ngram[ngram])} instances")
    print("-" * 50)
    
    c = Counter(high_level_labels_by_ngram[ngram])
    
    o = []
    for k in sorted(c.keys()):
        pct = c[k]/sum(c.values())*100
        print(f"  {str(k):35s} {c[k]:4d} ({pct:5.1f}%)")
        o.append([c[k]])
    observations.append(o)



HIGH-LEVEL LABELS BY N-GRAM LENGTH (ignoring Unsure/Ambiguous)

1-grams (single words): 1496 instances
--------------------------------------------------
  ('Non-scare quotes',)                474 ( 31.7%)
  ('Scare quotes',)                    806 ( 53.9%)
  ('Scare quotes', 'Non-scare quotes')  216 ( 14.4%)

+1-grams (multi-word phrases): 1004 instances
--------------------------------------------------
  ('Non-scare quotes',)                442 ( 44.0%)
  ('Scare quotes',)                    416 ( 41.4%)
  ('Scare quotes', 'Non-scare quotes')  146 ( 14.5%)


### Statistical Significance Testing: High-Level Labels by n-gram length

In [20]:
stat, pval, dfr, exp = chi2_contingency(observations, correction=True)

print("\n" + "=" * 70)
print("CHI-SQUARE TEST: High-level labels by n-gram length")
print("=" * 70)
print(f"Test statistic: {stat:.4f}")
print(f"P-value: {pval:.6f}")
print(f"Degrees of freedom: {dfr}")

if pval < 0.05:
    print("\n✓ STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("  → N-gram length significantly affects high-level labels")
else:
    print("\n✗ NOT SIGNIFICANT (p ≥ 0.05)")
print("=" * 70)


CHI-SQUARE TEST: High-level labels by n-gram length
Test statistic: 44.0004
P-value: 0.000000
Degrees of freedom: 2

✓ STATISTICALLY SIGNIFICANT (p < 0.05)
  → N-gram length significantly affects high-level labels


### Key Observations

There is a statistically significant difference in high-level labels by n-gram length:
- **1-grams (single words):** 54% are SQ 
- **+1-grams (multi-word):** 42% are SQ



In [21]:
fine_level_labels_by_ngram = {'1':[],"+1":[]}

# Without Unsure/Ambiguous/FLAG
for item in corpus:
    lbs = get_labels_one_level(item, FINE_LEVEL)
    lbs = tuple([lb for lb in lbs if lb not in ["Unsure", flag_label] and "Ambiguous" not in lb])
    ngram = '1' if item['data']['ngram'] == '1' else '+1'
    fine_level_labels_by_ngram[ngram].append(lbs)

observations_by_label = dict()

print("\n" + "=" * 70)
print("FINE-LEVEL LABELS BY N-GRAM LENGTH (ignoring Unsure/Ambiguous/FLAG)")
print("=" * 70)

for ngram in ['1', '+1']:
    ngram_name = "1-grams" if ngram == '1' else "+1-grams"
    print(f"\n{ngram_name}:")
    print("-" * 70)

    c = Counter(fine_level_labels_by_ngram[ngram])
    total_diff_individual_labels = sorted(set([lb for lbcombi in c.keys() for lb in lbcombi]))

    total = len([x for x in corpus if x['data']['ngram'] == '1']) if ngram == '1' \
            else len([x for x in corpus if x['data']['ngram'] != '1'])

    print(f"{'Label':<50} {'Pure':>6} {'Comb':>6} {'Total':>7} {'Pct':>7}")
    print("-" * 70)

    for lb in total_diff_individual_labels:
        if lb not in observations_by_label:
            observations_by_label[lb] = []

        pure = c.get((lb,), 0)
        combined = sum([c[k] for k in c.keys() if lb in k and k != (lb,)])
        total_lb = pure + combined
        pct = (total_lb/total*100) if total > 0 else 0

        print(f"{lb:<50} {pure:6d} {combined:6d} {total_lb:7d} {pct:6.1f}%")

        observations_by_label[lb].append([pure+combined, total-(pure+combined)])
    print()

# FLAG shown separately
print("-" * 70)
print("SQ FLAG (shown separately):")
print("-" * 70)
if flag_label not in observations_by_label:
    observations_by_label[flag_label] = []
for ngram in ['1', '+1']:
    ngram_name = "1-grams" if ngram == '1' else "+1-grams"
    total = len([x for x in corpus if x['data']['ngram'] == '1']) if ngram == '1' \
            else len([x for x in corpus if x['data']['ngram'] != '1'])
    flag_count = sum(1 for item in corpus
                     if (item['data']['ngram'] == '1') == (ngram == '1')
                     and flag_label in get_labels_one_level(item, FINE_LEVEL))
    pct = (flag_count / total * 100) if total > 0 else 0
    print(f"  {ngram_name:<12} {flag_count:4d} / {total:4d} ({pct:5.1f}%)")
    observations_by_label[flag_label].append([flag_count, total - flag_count])
print("=" * 70)


FINE-LEVEL LABELS BY N-GRAM LENGTH (ignoring Unsure/Ambiguous/FLAG)

1-grams:
----------------------------------------------------------------------
Label                                                Pure   Comb   Total     Pct
----------------------------------------------------------------------
NSQ: Designation                                       83     31     114    7.6%
NSQ: Direct quoting                                    35      3      38    2.5%
NSQ: Emphasis                                          25     18      43    2.9%
NSQ: Meta-linguistic comment                          312    181     493   33.0%
NSQ: Mixed quoting without attitude                     8      9      17    1.1%
SQ: Mixed quotation with attitude                      34      5      39    2.6%
SQ: Usage                                             750    206     956   63.9%
SQ: Word                                               22      5      27    1.8%


+1-grams:
--------------------------------------

In [22]:
print("\n" + "=" * 70)
print("PROPORTIONS: Fine-Level Labels by N-gram Length")
print("=" * 70)
print(f"\n{'Label':<45} {'1-gram %':>12} {'+1-gram %':>12}")
print("-" * 70)

for lb in sorted(observations_by_label.keys()):
    pct_1gram = observations_by_label[lb][0][0] / (observations_by_label[lb][0][0] + observations_by_label[lb][0][1]) * 100
    pct_plus1 = observations_by_label[lb][1][0] / (observations_by_label[lb][1][0] + observations_by_label[lb][1][1]) * 100
    
    print(f"{lb:<45} {pct_1gram:11.1f}% {pct_plus1:11.1f}%")


PROPORTIONS: Fine-Level Labels by N-gram Length

Label                                             1-gram %    +1-gram %
----------------------------------------------------------------------
NSQ: Designation                                      7.6%        27.2%
NSQ: Direct quoting                                   2.5%         7.9%
NSQ: Emphasis                                         2.9%         1.5%
NSQ: Meta-linguistic comment                         33.0%        22.3%
NSQ: Mixed quoting without attitude                   1.1%         2.5%
SQ: Echo Flag                                        25.9%        21.1%
SQ: Mixed quotation with attitude                     2.6%         4.1%
SQ: Usage                                            63.9%        50.6%
SQ: Word                                              1.8%         1.3%


In [23]:
print("\n" + "=" * 70)
print("SIGNIFICANCE TESTING: Fine-Level Labels by N-gram Length")
print("=" * 70)
print("\nChi-Square Tests (with Yates' correction):")
print("-" * 70)

significant_results = []

for lb in sorted(observations_by_label.keys()):
    stat, pval, dfr, exp = chi2_contingency(observations_by_label[lb], correction=True)
    
    if pval < 0.05:
        sign = "✓ SIGNIFICANT"
        significant_results.append(lb)
    else:
        sign = "✗ Not significant"
    
    print(f"{lb:<45} p={pval:.4f}  {sign}")

if significant_results:
    print("\n" + "=" * 70)
    print(f"⭐ {len(significant_results)} SIGNIFICANT RESULTS (p < 0.05):")
    print("=" * 70)
    for lb in significant_results:
        print(f"  • {lb}")


SIGNIFICANCE TESTING: Fine-Level Labels by N-gram Length

Chi-Square Tests (with Yates' correction):
----------------------------------------------------------------------
NSQ: Designation                              p=0.0000  ✓ SIGNIFICANT
NSQ: Direct quoting                           p=0.0000  ✓ SIGNIFICANT
NSQ: Emphasis                                 p=0.0347  ✓ SIGNIFICANT
NSQ: Meta-linguistic comment                  p=0.0000  ✓ SIGNIFICANT
NSQ: Mixed quoting without attitude           p=0.0154  ✓ SIGNIFICANT
SQ: Echo Flag                                 p=0.0073  ✓ SIGNIFICANT
SQ: Mixed quotation with attitude             p=0.0523  ✗ Not significant
SQ: Usage                                     p=0.0000  ✓ SIGNIFICANT
SQ: Word                                      p=0.4045  ✗ Not significant

⭐ 7 SIGNIFICANT RESULTS (p < 0.05):
  • NSQ: Designation
  • NSQ: Direct quoting
  • NSQ: Emphasis
  • NSQ: Meta-linguistic comment
  • NSQ: Mixed quoting without attitude
  • SQ: Echo Fla

---

## WMN Analysis: Loading NeWMe Data

Loading the NeWMe corpus to analyze instances that overlap with WMN spans.

In [24]:
# Load NeWMe corpus
newme_corpus = json.load(open(path_to_newme_json))
newme_reddit = [item for item in newme_corpus if item['data']['corpus'] == 'Reddit']

print(f"Loaded {len(newme_corpus)} NeWMe items ({len(newme_reddit)} Reddit instances)")

Loaded 8350 NeWMe items (3012 Reddit instances)


In [25]:
def find_short_quoted_phrases(text):
    """
    Identify quoted phrases in text, excluding those within citation markers.
    
    Searches for text enclosed in:
    - Double quotes: "..."
    - Single quotes: '...' (not inside words)
    - French quotes: «...»
    
    Args:
        text: String to search for quoted phrases
    
    Returns:
        Tuple of (all_quoted, short_phrases):
        - all_quoted: List of tuples (phrase, start_pos, end_pos) for all quotes
        - short_phrases: List of tuples for quotes containing 1-3 words only
    """
    # Step 1: Identify all [STA-CITE] ... [END-CITE] spans to exclude
    cite_spans = [(m.start(), m.end()) for m in re.finditer(
        r'\[STA-CITE\].*?\[END-CITE\]', text, flags=re.DOTALL
    )]

    def is_inside_cite(index):
        """Check if a character index is inside a citation span."""
        return any(start <= index < end for start, end in cite_spans)
    
    # Step 2: Define patterns for various types of quotes
    quote_patterns = [
        r'"([^"]*?)"',                # Double quotes
        r"(?<!\w)'([^']*?)'(?!\w)",   # Single quotes (not inside words)
        r'«\s*([^»]*?)\s*»',          # French quotes
    ]
    
    # Step 3: Apply all patterns and collect matches
    short_phrases = []
    all_quoted = []

    for pattern in quote_patterns:
        for match in re.finditer(pattern, text):
            phrase = match.group(1)
            start = match.start(1)
            end = match.end(1)

            # Skip if inside citation
            if is_inside_cite(start):
                continue

            all_quoted.append((phrase, start, end))

            # Keep only if 1–3 words
            if 1 <= len(phrase.strip().split()) <= 3:
                short_phrases.append((phrase, start, end))
                
    return all_quoted, short_phrases


In [26]:
# Extract quoted phrases from NeWMe Reddit spans
print("Extracting quoted phrases from NeWMe-Reddit spans...")
print("Note: This code works only with the Reddit portion of NeWMe because every utterance was annotated separately.")
print("It may not work with BNC where sometimes we annotated blocks of utterances as a single span.")

newme_reddit_quotes = []
newme_reddit_spans = 0


for item in newme_reddit:
    for result in item['annotations'][0]['result']:
        if 'text' in result['value'] and result['from_name'] != 'comment' and result['value']['paragraphlabels'] != ['Trigger reference']:
            if result['value']['start'] != result['value']['end']:
                print('Warning: Start != End found!')               
                
            utterance = item['data']['dialogue'][int(result['value']['start'])]['text']
            newme_reddit_spans += 1
            
            long_matches, matches = find_short_quoted_phrases(utterance)
                      
            utt_number1 = result['value']['start']
            utt_id = item['data']['dialogue'][int(result['value']['start'])]['id']
            conv_id = get_conv_id(item['data']['id'])           
            
            for match in matches:      
                spanstart, spanend = match[1], match[2]
                # inside the marked span in newme?
                spanstart, spanend = match[1], match[2]                    
                if spanstart >= result['value']['startOffset'] and spanend <= result['value']['endOffset']:                        
                    d = {'conv_id':conv_id, 'utt_id':utt_id, 'quoted_text': match[0], 'startspan': spanstart, 'endspan': spanend, 'utterance':utterance, 'span_type':[result['value']['paragraphlabels'][0]]}
                    d['quote_type'] = 'quotes'
                    newme_reddit_quotes.append(d)


### Note that one same usage may have been annotated as part of two separate spans. It will be counted double if we do not double check.
### Let's clean for that here.

for item in newme_reddit_quotes:
    item['already_added'] = False

new_newme_reddit_quotes = []
for i, item in enumerate(newme_reddit_quotes):    
    modified_item = deepcopy(item)
    if item['already_added']:
        continue
    for other_item in newme_reddit_quotes[i+1:]:        
        # if all keys are the same (except for span type which we dont care about), merge them into one!
        all_keys_the_same = True
        for k in other_item:
            if k != "span_type": 
                if other_item[k] != modified_item[k]:
                    all_keys_the_same = False
                    break
        if all_keys_the_same:
            modified_item['span_type'].append(other_item['span_type'][0])
            other_item['already_added'] = True
            
    new_newme_reddit_quotes.append(modified_item)
        
     
for item in new_newme_reddit_quotes:
    item['span_type'] = tuple(item['span_type'])


print("Newme reddit quotes:", len(newme_reddit_quotes))
print("Newme reddit quotes (cleaned):", len(new_newme_reddit_quotes))

newme_reddit_quotes = new_newme_reddit_quotes

print("Total number of newme reddit spans:", newme_reddit_spans)

print(f"Approximately {len(newme_reddit_quotes)/newme_reddit_spans*100:.1f}% of NeWMe-Reddit spans (2575 total) contain quotes")

Extracting quoted phrases from NeWMe-Reddit spans...
Note: This code works only with the Reddit portion of NeWMe because every utterance was annotated separately.
It may not work with BNC where sometimes we annotated blocks of utterances as a single span.
Newme reddit quotes: 773
Newme reddit quotes (cleaned): 699
Total number of newme reddit spans: 2575
Approximately 27.1% of NeWMe-Reddit spans (2575 total) contain quotes


#### Find the WMN coincidences and enrich them with WMN info

In [27]:
# Match corpus instances with WMN spans and enrich with WMN information
print("\nMatching corpus instances with WMN spans...")
coincidence = []
wmn_related_ids = []

for annotated_item in corpus:
    for quoted_newme in newme_reddit_quotes:
        if quoted_newme['conv_id'] == get_conv_id(annotated_item['data']['id']):            
            if quoted_newme['utt_id'] == "_".join(annotated_item['data']['id'].split("_")[2:4]):                
                if annotated_item['data']['quoted_passage'] == quoted_newme["quoted_text"]:                    
                    if annotated_item['data']['start'] == quoted_newme['startspan'] and annotated_item['data']['end'] == quoted_newme['endspan']:
                        coincidence.append((annotated_item, quoted_newme))
                        wmn_related_ids.append(annotated_item['data']['item_order'])
                        annotated_item['data']['wmn-span_type'] = quoted_newme['span_type']

print(f"Found {len(coincidence)} corpus instances overlapping with WMN spans")

# Summary statistics
wmn_count = len([item for item in corpus if 'wmn-span_type' in item['data']])
no_wmn_count = len(corpus) - wmn_count

print("\n" + "=" * 70)
print("WMN OVERLAP SUMMARY")
print("=" * 70)
print(f"Instances with WMN overlap:    {wmn_count:4d} ({wmn_count/len(corpus)*100:5.1f}%)")
print(f"Instances without WMN overlap: {no_wmn_count:4d} ({no_wmn_count/len(corpus)*100:5.1f}%)")
print(f"Total corpus:                  {len(corpus):4d}")
print("=" * 70)


Matching corpus instances with WMN spans...
Found 690 corpus instances overlapping with WMN spans

WMN OVERLAP SUMMARY
Instances with WMN overlap:     690 ( 27.6%)
Instances without WMN overlap: 1810 ( 72.4%)
Total corpus:                  2500


In [28]:
# WMN span type distribution
c_span_types = Counter([x['data']['wmn-span_type'] for x in corpus if 'wmn-span_type' in x['data']])

print("\n" + "=" * 70)
print("WMN SPAN TYPE DISTRIBUTION (RAW COUNTS)")
print("=" * 70)
for span_combo, count in c_span_types.most_common():
    print(f"  {str(span_combo):50s} {count:3d}")

print("\n" + "-" * 70)
print("INSTANCES BY PRIMARY SPAN TYPE:")
print("-" * 70)
for span_type in ["Trigger", "Indicator", "Negotiation"]:
    count = sum([c_span_types[k] for k in c_span_types if span_type in k])
    print(f"  {span_type:12s} {count:3d}")
print("=" * 70)


WMN SPAN TYPE DISTRIBUTION (RAW COUNTS)
  ('Negotiation',)                                   372
  ('Indicator',)                                     208
  ('Trigger',)                                        46
  ('Negotiation', 'Negotiation')                      28
  ('Trigger', 'Trigger')                              23
  ('Indicator', 'Indicator')                           4
  ('Trigger', 'Trigger', 'Trigger')                    4
  ('Indicator', 'Negotiation')                         2
  ('Negotiation', 'Indicator')                         2
  ('Negotiation', 'Trigger', 'Negotiation')            1

----------------------------------------------------------------------
INSTANCES BY PRIMARY SPAN TYPE:
----------------------------------------------------------------------
  Trigger       74
  Indicator    216
  Negotiation  405


---

## WMN Overlap - High-Level Label Distribution

Comparing label distributions between instances that overlap with WMN spans and those that don't.

In [29]:
# Prepare data structures for high-level label analysis
high_level_labels_wmn = []
high_level_labels_nowmn = []
high_level_labels_wmn_by_span = {'Trigger': [], 'Indicator': [], 'Negotiation': []}

# Categorize instances by WMN overlap
for item in corpus:    
    lbs = get_labels_one_level(item, HIGH_LEVEL)
    lbs = tuple([lb for lb in lbs if lb not in ["Unsure"] and "Ambiguous" not in lb])    
    
    if 'wmn-span_type' in item['data']:    
        span_types = item['data']['wmn-span_type']
        high_level_labels_wmn.append(lbs)
        for st in span_types:            
            high_level_labels_wmn_by_span[st].append(lbs)            
    else:
        high_level_labels_nowmn.append(lbs)

# Prepare observations for chi-square test
observations = []

# WMN instances
print("\n" + "=" * 70)
print("HIGH-LEVEL LABELS: WMN vs Non-WMN Instances")
print("=" * 70)
print("\n🔹 WMN-Related Instances:")
print("-" * 70)
c_wmn = Counter(high_level_labels_wmn)

o_wmn = []
for label_combo in sorted(c_wmn.keys()):    
    count = c_wmn[label_combo]
    pct = (count / len(high_level_labels_wmn) * 100) if high_level_labels_wmn else 0
    print(f"  {str(label_combo):45s} {count:4d} ({pct:5.1f}%)")
    o_wmn.append(count)
observations.append(o_wmn)

# Non-WMN instances
print("\n🔹 Non-WMN Instances:")
print("-" * 70)
cn_nowmn = Counter(high_level_labels_nowmn)

o_nowmn = []
for label_combo in sorted(cn_nowmn.keys()):    
    count = cn_nowmn[label_combo]
    pct = (count / len(high_level_labels_nowmn) * 100) if high_level_labels_nowmn else 0
    print(f"  {str(label_combo):45s} {count:4d} ({pct:5.1f}%)")
    o_nowmn.append(count)
observations.append(o_nowmn)

# By span type
print("\n" + "=" * 70)
print("HIGH-LEVEL LABELS BY WMN SPAN TYPE")
print("=" * 70)

for span_type in ['Trigger', 'Indicator', 'Negotiation']:
    print(f"\n{span_type}:")
    print("-" * 70)
    
    c_span = Counter(high_level_labels_wmn_by_span[span_type])
    total_span = len(high_level_labels_wmn_by_span[span_type])
    
    if total_span == 0:
        print("  No instances")
        continue
    
    for label_combo in sorted(c_span.keys()):
        count = c_span[label_combo]
        pct = (count / total_span * 100) if total_span > 0 else 0
        print(f"  {str(label_combo):45s} {count:4d} ({pct:5.1f}%)")

print("\n" + "=" * 70)


HIGH-LEVEL LABELS: WMN vs Non-WMN Instances

🔹 WMN-Related Instances:
----------------------------------------------------------------------
  ('Non-scare quotes',)                          309 ( 44.8%)
  ('Scare quotes',)                              226 ( 32.8%)
  ('Scare quotes', 'Non-scare quotes')           155 ( 22.5%)

🔹 Non-WMN Instances:
----------------------------------------------------------------------
  ('Non-scare quotes',)                          607 ( 33.5%)
  ('Scare quotes',)                              996 ( 55.0%)
  ('Scare quotes', 'Non-scare quotes')           207 ( 11.4%)

HIGH-LEVEL LABELS BY WMN SPAN TYPE

Trigger:
----------------------------------------------------------------------
  ('Non-scare quotes',)                           30 ( 28.6%)
  ('Scare quotes',)                               57 ( 54.3%)
  ('Scare quotes', 'Non-scare quotes')            18 ( 17.1%)

Indicator:
----------------------------------------------------------------------
  ('Non

In [30]:
# Proportion comparison: WMN vs Non-WMN
print("\n" + "=" * 70)
print("PROPORTIONS: WMN vs Non-WMN")
print("=" * 70)
print(f"\n{'Label':45s} {'WMN %':>12} {'Non-WMN %':>12}")
print("-" * 70)

# Assuming observations has [WMN row, Non-WMN row] with values in same order as sorted keys
label_keys = sorted(c_wmn.keys())  # Use the keys from WMN counter

for i, label in enumerate(label_keys):
    wmn_prop = observations[0][i] / sum(observations[0]) * 100
    nowmn_prop = observations[1][i] / sum(observations[1]) * 100
    print(f"{str(label):45s} {wmn_prop:11.1f}% {nowmn_prop:11.1f}%")

print("=" * 70)


PROPORTIONS: WMN vs Non-WMN

Label                                                WMN %    Non-WMN %
----------------------------------------------------------------------
('Non-scare quotes',)                                44.8%        33.5%
('Scare quotes',)                                    32.8%        55.0%
('Scare quotes', 'Non-scare quotes')                 22.5%        11.4%


#### Significance testing

In [31]:
stat, pval, dfr, exp = chi2_contingency(observations, correction=True)

if pval < 0.05:
    print("\n✓ STATISTICALLY SIGNIFICANT (p < 0.05)")
    print("  → High-level quote distribution is significantly different between WMN and non-WMN instances")
else:
    print("\n✗ NOT SIGNIFICANT (p ≥ 0.05)")



✓ STATISTICALLY SIGNIFICANT (p < 0.05)
  → High-level quote distribution is significantly different between WMN and non-WMN instances


Observation: significant result. Those involving WMNs are more often NSQ or both, whereas the others are more often sq.

In [32]:
fine_level_labels_wmn = []
fine_level_labels_nowmn = []
fine_level_labels_wmn_by_span = {'Trigger':[], 'Indicator':[], 'Negotiation':[]}

# Exclude FLAG labels for this analysis
for item in corpus:    
    lbs = get_labels_one_level(item, FINE_LEVEL)
    lbs = tuple([lb for lb in lbs if lb not in ["Unsure"] and "Ambiguous" not in lb and "FLAG" not in lb])    
    if 'wmn-span_type' in item['data']:    
        span_types = item['data']['wmn-span_type']
        fine_level_labels_wmn.append(lbs)
        for st in span_types:            
            fine_level_labels_wmn_by_span[st].append(lbs)
    else:
        fine_level_labels_nowmn.append(lbs)
            
            
print("\n" + "=" * 70)
print("FINE-LEVEL ANALYSIS: WMN vs NON-WMN")
print("=" * 70)

print("\n📊 WMN-Related Instances:")
c = Counter(fine_level_labels_wmn)
  
observations_by_label = dict()
total_diff_individual_labels = set([lb for lbcombi in c.keys() for lb in lbcombi])

for lb in total_diff_individual_labels:
    if lb not in observations_by_label:
        observations_by_label[lb] = []

    total = len(fine_level_labels_wmn)
    pure = c.get((lb,), 0)
    combined = sum([c[k] for k in c.keys() if lb in k and k != (lb,)])
    pct = (pure+combined)/len(fine_level_labels_wmn)*100 if len(fine_level_labels_wmn) > 0 else 0
    print(f"  {lb:50s} pure: {pure:3d} | combined: {combined:3d} | total: {pure+combined:3d} ({pct:5.1f}%)")
    observations_by_label[lb].append([pure+combined, total-(pure+combined)])
   
cn = Counter(fine_level_labels_nowmn)

print("\n📊 Non-WMN Instances:")

for lb in total_diff_individual_labels:   
    total = len(fine_level_labels_nowmn)
    pure = cn.get((lb,), 0)
    combined = sum([cn[k] for k in cn.keys() if lb in k and k != (lb,)])  # BUG FIX: changed from c.keys() to cn.keys()
    pct = (pure+combined)/len(fine_level_labels_nowmn)*100 if len(fine_level_labels_nowmn) > 0 else 0
    print(f"  {lb:50s} pure: {pure:3d} | combined: {combined:3d} | total: {pure+combined:3d} ({pct:5.1f}%)")
    observations_by_label[lb].append([pure+combined, total-(pure+combined)])
    
    
print("\n" + "-" * 70)   
print("BY WMN SPAN TYPE:")
print("-" * 70)

for sign in fine_level_labels_wmn_by_span:
    print(f"\n  {sign}:")
    
    c_span = Counter(fine_level_labels_wmn_by_span[sign])
    
    for lb in total_diff_individual_labels:
        pure = c_span.get((lb,), 0)
        combined = sum([c_span[k] for k in c_span.keys() if lb in k and k != (lb,)])
        pct = (pure+combined)/len(fine_level_labels_wmn_by_span[sign])*100 if len(fine_level_labels_wmn_by_span[sign]) > 0 else 0
        if pure + combined > 0:  # Only show labels that appear
            print(f"    {lb:48s} pure: {pure:3d} | combined: {combined:3d} ({pct:5.1f}%)")
    print()


print("number of no wmn instances in total:", len(fine_level_labels_nowmn))


FINE-LEVEL ANALYSIS: WMN vs NON-WMN

📊 WMN-Related Instances:
  SQ: Word                                           pure:   5 | combined:   1 | total:   6 (  0.9%)
  NSQ: Meta-linguistic comment                       pure: 232 | combined: 148 | total: 380 ( 55.1%)
  NSQ: Emphasis                                      pure:   7 | combined:   8 | total:  15 (  2.2%)
  NSQ: Designation                                   pure:  33 | combined:  13 | total:  46 (  6.7%)
  SQ: Mixed quotation with attitude                  pure:  10 | combined:   3 | total:  13 (  1.9%)
  NSQ: Mixed quoting without attitude                pure:   2 | combined:   1 | total:   3 (  0.4%)
  SQ: Usage                                          pure: 103 | combined: 259 | total: 362 ( 52.5%)
  NSQ: Direct quoting                                pure:  28 | combined:   3 | total:  31 (  4.5%)
  SQ: Echo Flag                                      pure:   0 | combined: 239 | total: 239 ( 34.6%)

📊 Non-WMN Instances:
  SQ: 

In [33]:
# Modify this cell to make it print the result more nicely as in other cells.


for lb in observations_by_label:
    stat, pval, dfr, exp = chi2_contingency(observations_by_label[lb], correction=True)
    significance = "✓ SIGNIFICANT" if pval < 0.05 else "✗ Not significant"

    print("p-value", lb, ":", pval, significance)


p-value SQ: Word : 0.10548124639639384 ✗ Not significant
p-value NSQ: Meta-linguistic comment : 3.61211600591503e-72 ✓ SIGNIFICANT
p-value NSQ: Emphasis : 0.8799904931606977 ✗ Not significant
p-value NSQ: Designation : 8.645649474365228e-14 ✓ SIGNIFICANT
p-value SQ: Mixed quotation with attitude : 0.02917408458713311 ✓ SIGNIFICANT
p-value NSQ: Mixed quoting without attitude : 0.004847489490656519 ✓ SIGNIFICANT
p-value SQ: Usage : 0.00016002917719107332 ✓ SIGNIFICANT
p-value NSQ: Direct quoting : 0.8667635846503835 ✗ Not significant
p-value SQ: Echo Flag : 1.716411745268134e-14 ✓ SIGNIFICANT



Analyzing instances with the Echo flag separately:

In [34]:
# Filter for FLAG label instances only
flag_label = "SQ: Echo Flag"

fine_level_labels_wmn_flag = []
fine_level_labels_nowmn_flag = []
fine_level_labels_wmn_by_span_flag = {'Trigger': [], 'Indicator': [], 'Negotiation': []}

# Count total WMN and non-WMN instances
total_wmn_all = len([item for item in corpus if 'wmn-span_type' in item['data']])
total_nowmn_all = len([item for item in corpus if 'wmn-span_type' not in item['data']])

# Count instances with FLAG label
total_with_flag = 0

for item in corpus:    
    lbs = get_labels_one_level(item, FINE_LEVEL)
    if flag_label not in lbs:
        continue
    
    total_with_flag += 1
    lbs = tuple(sorted([lb for lb in lbs if lb not in ["Unsure"] and "Ambiguous" not in lb]))
    
    if 'wmn-span_type' in item['data']:    
        span_types = item['data']['wmn-span_type']
        fine_level_labels_wmn_flag.append(lbs)
        for st in span_types:            
            fine_level_labels_wmn_by_span_flag[st].append(lbs)
    else:
        fine_level_labels_nowmn_flag.append(lbs)

print("\n" + "=" * 70)
print(f"ECHO FLAG ANALYSIS: '{flag_label}'")
print("=" * 70)
print(f"\n📊 Total instances with FLAG label: {total_with_flag} ({total_with_flag/len(corpus)*100:.1f}% of corpus)")

# WMN-related instances with FLAG
total_flag_wmn = len(fine_level_labels_wmn_flag)
pct_flag_in_wmn = (total_flag_wmn / total_wmn_all * 100) if total_wmn_all > 0 else 0

print(f"\n🔹 WMN-Related Instances: {total_wmn_all} total")
print(f"  FLAG instances:        {total_flag_wmn:4d} ({pct_flag_in_wmn:5.1f}% of all WMN instances)")

c_wmn = Counter(fine_level_labels_wmn_flag)
pure_wmn = c_wmn.get((flag_label,), 0)
combined_wmn = total_flag_wmn - pure_wmn

print(f"    • Pure (FLAG only):    {pure_wmn:4d}")
print(f"    • Combined (FLAG + X): {combined_wmn:4d}")

observations_by_label = {flag_label: []}
observations_by_label[flag_label].append([total_flag_wmn, total_wmn_all - total_flag_wmn])

# Non-WMN instances with FLAG
total_flag_nowmn = len(fine_level_labels_nowmn_flag)
pct_flag_in_nowmn = (total_flag_nowmn / total_nowmn_all * 100) if total_nowmn_all > 0 else 0

print(f"\n🔹 Non-WMN Instances: {total_nowmn_all} total")
print(f"  FLAG instances:        {total_flag_nowmn:4d} ({pct_flag_in_nowmn:5.1f}% of all non-WMN instances)")

cn_nowmn = Counter(fine_level_labels_nowmn_flag)
pure_nowmn = cn_nowmn.get((flag_label,), 0)
combined_nowmn = total_flag_nowmn - pure_nowmn

print(f"    • Pure (FLAG only):    {pure_nowmn:4d}")
print(f"    • Combined (FLAG + X): {combined_nowmn:4d}")

observations_by_label[flag_label].append([total_flag_nowmn, total_nowmn_all - total_flag_nowmn])

# Breakdown by WMN span type
print("\n" + "-" * 70)
print("FLAG DISTRIBUTION BY WMN SPAN TYPE:")
print("-" * 70)

for span_type in ['Trigger', 'Indicator', 'Negotiation']:
    span_instances = fine_level_labels_wmn_by_span_flag[span_type]
    total_span_all = len(high_level_labels_wmn_by_span[span_type])
    
    if total_span_all == 0:
        print(f"\n  {span_type}: No instances of this span type")
        continue
    
    total_flag_span = len(span_instances)
    pct_flag_span = (total_flag_span / total_span_all * 100) if total_span_all > 0 else 0
    
    print(f"\n  {span_type}: {total_span_all} total instances")
    print(f"    FLAG instances: {total_flag_span:3d} ({pct_flag_span:5.1f}% of {span_type} instances)")
    
    if total_flag_span > 0:
        c_span = Counter(span_instances)
        pure_span = c_span.get((flag_label,), 0)
        combined_span = total_flag_span - pure_span
        print(f"      • Pure (FLAG only):    {pure_span:3d}")
        print(f"      • Combined (FLAG + X): {combined_span:3d}")

print("\n" + "=" * 70)


ECHO FLAG ANALYSIS: 'SQ: Echo Flag'

📊 Total instances with FLAG label: 599 (24.0% of corpus)

🔹 WMN-Related Instances: 690 total
  FLAG instances:         239 ( 34.6% of all WMN instances)
    • Pure (FLAG only):       0
    • Combined (FLAG + X):  239

🔹 Non-WMN Instances: 1810 total
  FLAG instances:         360 ( 19.9% of all non-WMN instances)
    • Pure (FLAG only):       0
    • Combined (FLAG + X):  360

----------------------------------------------------------------------
FLAG DISTRIBUTION BY WMN SPAN TYPE:
----------------------------------------------------------------------

  Trigger: 105 total instances
    FLAG instances:  14 ( 13.3% of Trigger instances)
      • Pure (FLAG only):      0
      • Combined (FLAG + X):  14

  Indicator: 220 total instances
    FLAG instances: 106 ( 48.2% of Indicator instances)
      • Pure (FLAG only):      0
      • Combined (FLAG + X): 106

  Negotiation: 434 total instances
    FLAG instances: 129 ( 29.7% of Negotiation instances)
   

In [35]:
# Calculate WMN span type statistics
triggers = len([x for x in corpus if 'wmn-span_type' in x['data'] and 'Trigger' in x['data']['wmn-span_type']])
indicators = len([x for x in corpus if 'wmn-span_type' in x['data'] and 'Indicator' in x['data']['wmn-span_type']])
negotiations = len([x for x in corpus if 'wmn-span_type' in x['data'] and 'Negotiation' in x['data']['wmn-span_type']])

print("\n" + "=" * 70)
print("WMN SPAN TYPE DISTRIBUTION")
print("=" * 70)
print(f"Instances involving Triggers:     {triggers:3d}")
print(f"Instances involving Indicators:   {indicators:3d}")
print(f"Instances involving Negotiations: {negotiations:3d}")
print("=" * 70)


WMN SPAN TYPE DISTRIBUTION
Instances involving Triggers:      74
Instances involving Indicators:   216
Instances involving Negotiations: 405


In [36]:
# Statistical significance test for FLAG label
print("\n" + "=" * 70)
print("CHI-SQUARE TEST: FLAG Label (WMN vs Non-WMN)")
print("=" * 70)

for lb in observations_by_label:
    stat, pval, dfr, exp = chi2_contingency(observations_by_label[lb], correction=True)
    
    print(f"Test statistic: {stat:.4f}")
    print(f"P-value: {pval:.6f}")
    print(f"Degrees of freedom: {dfr}")
    
    if pval < 0.05:
        print("\n✓ STATISTICALLY SIGNIFICANT (p < 0.05)")
        print("  → FLAG label significantly more common in WMN-related instances")
    else:
        print("\n✗ NOT SIGNIFICANT (p ≥ 0.05)")

print("=" * 70)


CHI-SQUARE TEST: FLAG Label (WMN vs Non-WMN)
Test statistic: 58.8330
P-value: 0.000000
Degrees of freedom: 1

✓ STATISTICALLY SIGNIFICANT (p < 0.05)
  → FLAG label significantly more common in WMN-related instances


In [37]:
# Calculate proportions for comparison
print("\n" + "-" * 70)
print("PROPORTION COMPARISON:")
print("-" * 70)

for lb in observations_by_label:
    wmn_prop = observations_by_label[lb][0][0] / (observations_by_label[lb][0][0] + observations_by_label[lb][0][1])
    nowmn_prop = observations_by_label[lb][1][0] / (observations_by_label[lb][1][0] + observations_by_label[lb][1][1])
    
    print(f"\n{lb}:")
    print(f"  WMN instances:      ({wmn_prop*100:.1f}%)")
    print(f"  Non-WMN instances:  ({nowmn_prop*100:.1f}%)")    


----------------------------------------------------------------------
PROPORTION COMPARISON:
----------------------------------------------------------------------

SQ: Echo Flag:
  WMN instances:      (34.6%)
  Non-WMN instances:  (19.9%)


---

### Label Combinations in WMN Instances

Examining which label combinations appear in instances that overlap with WMN spans, broken down by span type.


In [38]:
print("\n" + "=" * 70)
print("LABEL COMBINATIONS IN WMN INSTANCES")
print("=" * 70)

c_wmn_combos = Counter(fine_level_labels_wmn)

combis_list = [(k, c_wmn_combos[k]) for k in c_wmn_combos if len(k) > 1]
combis_list.sort(key=lambda x: x[1], reverse=True)
combis_number = sum([count for _, count in combis_list])

print(f"\n📊 Multiple-label instances: {combis_number}", end="")
if fine_level_labels_wmn:
    print(f" ({combis_number/len(fine_level_labels_wmn)*100:.1f}% of WMN instances)")
else:
    print()

print(f"\n{'Combination':60s} {'Count':>6}")
print("-" * 70)
for combo, count in combis_list:
    pct = count / len(fine_level_labels_wmn) * 100 if fine_level_labels_wmn else 0
    print(f"  {str(combo):58s} {count:4d} ({pct:4.1f}%)")

# By span type
print("\n" + "-" * 70)
print("COMBINATIONS BY WMN SPAN TYPE:")
print("-" * 70)

for span in ['Trigger', 'Indicator', 'Negotiation']:
    print(f"\n  {span}:")
    c_span = Counter(fine_level_labels_wmn_by_span[span])
    combos_span = sorted(
        [(k, c_span[k]) for k in c_span if len(k) > 1],
        key=lambda x: x[1], reverse=True
    )
    if not combos_span:
        print("    No multiple-label combinations")
    else:
        total_span = len(fine_level_labels_wmn_by_span[span])
        for combo, count in combos_span:
            pct = count / total_span * 100 if total_span > 0 else 0
            print(f"    {str(combo):56s} {count:4d} ({pct:4.1f}%)")

print("\n" + "=" * 70)


LABEL COMBINATIONS IN WMN INSTANCES

📊 Multiple-label instances: 270 (39.1% of WMN instances)

Combination                                                   Count
----------------------------------------------------------------------
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Meta-linguistic comment')  118 (17.1%)
  ('SQ: Usage', 'SQ: Echo Flag')                              107 (15.5%)
  ('SQ: Usage', 'NSQ: Meta-linguistic comment')                17 ( 2.5%)
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Designation')            5 ( 0.7%)
  ('NSQ: Meta-linguistic comment', 'NSQ: Designation')          4 ( 0.6%)
  ('SQ: Usage', 'NSQ: Emphasis')                                3 ( 0.4%)
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Emphasis')               3 ( 0.4%)
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Meta-linguistic comment', 'NSQ: Designation')    1 ( 0.1%)
  ('SQ: Usage', 'SQ: Echo Flag', 'NSQ: Meta-linguistic comment', 'NSQ: Direct quoting')    1 ( 0.1%)
  ('SQ: Usage', 'NSQ: Designation')         

---

### N-gram Distribution by Quote Type and WMN

Examining how single vs. double quotes differ in their use of unigrams, and how WMN-related instances differ from the rest in terms of n-gram length.

In [39]:
annotated_ids = set([item['data']['item_order'] for item in corpus])
annotated_instances_info = [r for _, r in quotes_all_info.iterrows() if r['item_order'] in annotated_ids]
wmn_involved_ids = [x['data']['item_order'] for x in corpus if 'wmn-span_type' in x['data']]
ngram_type = [r['ngram'] for r in annotated_instances_info if r['item_order'] in wmn_involved_ids]

c = Counter(ngram_type)

print("\n" + "=" * 70)
print("N-GRAM TYPES IN WMN INSTANCES")
print("=" * 70)
for ngram, count in c.most_common():
    pct = count / len(ngram_type) * 100 if ngram_type else 0
    print(f"  {ngram:20s} {count:3d} ({pct:5.1f}%)")
print("=" * 70)

spans = [x['data']['wmn-span_type'] for x in corpus if 'wmn-span_type' in x['data']]
spans_and_ngrams = list(zip(spans, ngram_type))

print("\n" + "=" * 70)
print("N-GRAM TYPES IN WMN INSTANCES, BY SPAN TYPE")
print("=" * 70)
for n in [1, 2, 3]:
    for s in ["Trigger", "Indicator", "Negotiation"]:
        count = len([x for x in spans_and_ngrams if s in x[0] and x[1] == str(n)])
        if count > 0:
            print(f"  {s:12s} {n}-gram: {count:3d}")
print("=" * 70)


N-GRAM TYPES IN WMN INSTANCES
  1                    485 ( 70.3%)
  2                    149 ( 21.6%)
  3                     56 (  8.1%)

N-GRAM TYPES IN WMN INSTANCES, BY SPAN TYPE
  Trigger      1-gram:  52
  Indicator    1-gram: 160
  Negotiation  1-gram: 276
  Trigger      2-gram:  17
  Indicator    2-gram:  37
  Negotiation  2-gram:  96
  Trigger      3-gram:   5
  Indicator    3-gram:  19
  Negotiation  3-gram:  33


In [40]:
# N-gram analysis by span type
spans = [x['data']['wmn-span_type'] for x in corpus if 'wmn-span_type' in x['data']]
spans_and_ngrams = list(zip(spans, ngram_type))

print("\n" + "=" * 70)
print("N-GRAM TYPES BY WMN SPAN TYPE")
print("=" * 70)
for n in [1, 2, 3]:
    for span_type in ["Trigger", "Indicator", "Negotiation"]:
        count = len([x for x in spans_and_ngrams if span_type in x[0] and x[1] == str(n)])
        if count > 0:  # Only show non-zero counts
            print(f"  {span_type:12s} {n}-gram: {count:3d}")
print("=" * 70)


N-GRAM TYPES BY WMN SPAN TYPE
  Trigger      1-gram:  52
  Indicator    1-gram: 160
  Negotiation  1-gram: 276
  Trigger      2-gram:  17
  Indicator    2-gram:  37
  Negotiation  2-gram:  96
  Trigger      3-gram:   5
  Indicator    3-gram:  19
  Negotiation  3-gram:  33


In [41]:
# N-gram analysis for WMN instances
annotated_ids = set([item['data']['item_order'] for item in corpus])
annotated_instances_info = [r for _, r in quotes_all_info.iterrows() if r['item_order'] in annotated_ids]
wmn_involved_ids = [x['data']['item_order'] for x in corpus if 'wmn-span_type' in x['data']]
ngram_type = [r['ngram'] for r in annotated_instances_info if r['item_order'] in wmn_involved_ids]

c_ngram = Counter(ngram_type)

print("\n" + "=" * 70)
print("N-GRAM TYPES IN WMN INSTANCES")
print("=" * 70)
for ngram, count in c_ngram.most_common():
    pct = count / len(ngram_type) * 100 if ngram_type else 0
    print(f"  {ngram:20s} {count:3d} ({pct:5.1f}%)")
print("=" * 70)


N-GRAM TYPES IN WMN INSTANCES
  1                    485 ( 70.3%)
  2                    149 ( 21.6%)
  3                     56 (  8.1%)


In [42]:
single_quotes = [item for item in corpus if item['data']['sign'] == "'"]
double_quotes = [item for item in corpus if item['data']['sign'] == '"']

single_unigrams = len([item for item in single_quotes if item['data']['ngram'] == '1'])
double_unigrams = len([item for item in double_quotes if item['data']['ngram'] == '1'])

pct_single = (single_unigrams / len(single_quotes) * 100) if single_quotes else 0
pct_double = (double_unigrams / len(double_quotes) * 100) if double_quotes else 0

print("\n" + "=" * 70)
print("UNIGRAM USAGE BY QUOTE TYPE")
print("=" * 70)
print(f"Single quotes ('): {single_unigrams:4d} / {len(single_quotes):4d} are unigrams ({pct_single:5.1f}%)")
print(f"Double quotes (\"): {double_unigrams:4d} / {len(double_quotes):4d} are unigrams ({pct_double:5.1f}%)")
print("=" * 70)


UNIGRAM USAGE BY QUOTE TYPE
Single quotes ('):  210 /  310 are unigrams ( 67.7%)
Double quotes ("): 1286 / 2190 are unigrams ( 58.7%)
